<a href="https://colab.research.google.com/github/szh141/UVA-AMF/blob/main/OrganoSeg_Python_20260310.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

OrganoSeg matlab code adaptation

In [ ]:
import numpy as np
from skimage import io, img_as_float
from skimage.filters import threshold_otsu
from scipy.ndimage import convolve

def multiscale_adaptive_threshold(
        image,
        max_window=250,
        step=10,
        start_window=20,
        otsu_param=1.0):

    # ensure float
    img = img_as_float(image)

    h, w = img.shape
    combined = np.zeros((h, w), dtype=np.float32)

    for win in range(start_window, max_window + 1, step):

        # average filter kernel
        kernel = np.ones((win, win)) / (win * win)

        # smoothed image (replicate boundary)
        smoothed = convolve(img, kernel, mode='nearest')

        # difference image
        diff = smoothed - img

        # Otsu threshold
        t = threshold_otsu(diff) * otsu_param

        # binarize
        binary = diff > t

        # accumulate stack
        combined += binary.astype(np.float32)

    # final binary result
    result = combined > 0

    return result, combined


# example usage
image = io.imread("/content/drive/MyDrive/Colab images/spheroid.png", as_gray=True)

binary, stack_sum = multiscale_adaptive_threshold(
    image,
    max_window=250,
    step=10,
    otsu_param=1.0
)

io.imshow(binary)